# Stage 9 - interactive visualization

Reusable matching and boundary-filtered endpoint preparation remain in `src/`.
This notebook owns Napari layer styling, visibility, labels, and diagnostic controls.


In [ ]:
import numpy as np
import pandas as pd
from importlib import import_module, reload

from src.api import prepare_visualization_data
from src.io import (
    PipelinePaths,
    load_npy_time_series,
    load_processed_dataset_inputs,
    load_stage8_outputs,
    open_sample,
)

SAMPLE_ID = "44b6_0113de3b"
BOUNDARY_MARGIN_UM = 4.0
SHOW_BOUNDARY_TRACKS = False
SCENE_PADDING_ZYX = (2, 12, 12)

paths = PipelinePaths.discover()
sample_path = paths.sample_zarr(SAMPLE_ID)
inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
outputs = load_stage8_outputs(paths=paths)

cells = pd.concat(
    [frame.assign(frame=index) for index, frame in enumerate(inputs.time_frames)],
    ignore_index=True,
)
raw = open_sample(sample_path)
preprocessed, _ = load_npy_time_series(inputs.root / "preprocessing")
binary_mask, _ = load_npy_time_series(inputs.root / "masking")
instance_labels, _ = load_npy_time_series(inputs.root / "segmentation")

visualization = prepare_visualization_data(
    outputs.tracks,
    cells,
)
endpoint_helpers = reload(
    import_module("src.09_visualization.step02_endpoints")
)
VOXEL_SIZE_ZYX = endpoint_helpers.VOXEL_SIZE_ZYX
endpoint_groups = endpoint_helpers.prepare_endpoint_track_groups(
    visualization.tracks,
    cells,
    raw.shape[-3:],
    voxel_size_zyx=VOXEL_SIZE_ZYX,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)
SCALE_TZYX = (1.0, *VOXEL_SIZE_ZYX)

print("Voxel size ZYX:", VOXEL_SIZE_ZYX)
print("New failure candidates:", endpoint_groups.new_failure_tracks.track_id.nunique())
print("Ended failure candidates:", endpoint_groups.ended_failure_tracks.track_id.nunique())
print("Boundary entries:", endpoint_groups.boundary_entry_tracks.track_id.nunique())
print("Boundary exits:", endpoint_groups.boundary_exit_tracks.track_id.nunique())


In [ ]:
import napari
from diagnostics.cell_volume_extraction import add_cell_volume_extractor
from diagnostics.tracking_scene_extraction import add_tracking_scene_extractor

napari_layers = reload(import_module("src.09_visualization.napari_layers"))
add_track_group = napari_layers.add_track_group


viewer = napari.Viewer(ndisplay=3)
raw_contrast_limits = [
    float(np.percentile(raw, 1)),
    float(np.percentile(raw, 99.8)),
]
viewer.add_image(
    raw,
    name="Raw Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=raw_contrast_limits,
)
viewer.add_image(
    preprocessed,
    name="Preprocessed Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=(0.0, 1.0),
    visible=False,
)
viewer.add_labels(
    binary_mask,
    name="Binary Mask",
    scale=SCALE_TZYX,
    visible=False,
)
viewer.add_labels(
    instance_labels,
    name="Instance Labels",
    scale=SCALE_TZYX,
    visible=False,
)

all_tracks_layer = viewer.add_tracks(
    visualization.tracks_array,
    name="Tracks - all",
    scale=SCALE_TZYX,
    tail_length=20,
)
all_tracks_layer.visible = False
all_centers_layer = viewer.add_points(
    visualization.points_array,
    name="Centroids - all",
    scale=SCALE_TZYX,
    size=4,
    face_color="red",
    properties={
        "track_id": visualization.track_ids,
        "cell_id": visualization.tracks["cell_id"].to_numpy(),
    },
    text={
        "string": "{cell_id}",
        "size": 8,
        "color": "white",
        "anchor": "center",
    },
)
all_centers_layer.visible = False

add_track_group(
    viewer,
    endpoint_groups.ended_failure_tracks,
    track_name="Ended Tracks",
    point_name="Ended Centroids",
    color="red",
    scale=SCALE_TZYX,
)
add_track_group(
    viewer,
    endpoint_groups.new_failure_tracks,
    track_name="New Tracks",
    point_name="New Centroids",
    color="lime",
    scale=SCALE_TZYX,
)
add_track_group(
    viewer,
    endpoint_groups.boundary_entry_tracks,
    track_name="Boundary Entry Tracks",
    point_name="Boundary Entry Centroids",
    color="cyan",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)
add_track_group(
    viewer,
    endpoint_groups.boundary_exit_tracks,
    track_name="Boundary Exit Tracks",
    point_name="Boundary Exit Centroids",
    color="orange",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)


In [ ]:
scene_extractor = add_tracking_scene_extractor(
    viewer=viewer,
    cells=cells,
    instance_labels_volume=instance_labels,
    binary_mask_volume=binary_mask,
    image_volumes={"raw": raw, "preprocessed": preprocessed},
    sample_id=SAMPLE_ID,
    save_root=paths.tracking_scenes,
    voxel_size_zyx=VOXEL_SIZE_ZYX,
    default_padding_zyx=SCENE_PADDING_ZYX,
    cell_id_column="cell_id",
    frame_column="frame",
    source_metadata={
        "processed_dir": inputs.root,
        "cells_dir": inputs.root / "cells",
        "tracks_csv": paths.stage8_stitching / "tracks.csv",
        "source_zarr_array": paths.sample_zarr_array(SAMPLE_ID),
    },
)
cell_extractor = add_cell_volume_extractor(
    viewer=viewer,
    cells=cells,
    image_volume=raw,
    sample_id=SAMPLE_ID,
    preprocessed_volume=preprocessed,
    binary_mask_volume=binary_mask,
    instance_labels_volume=instance_labels,
    voxel_size_zyx=VOXEL_SIZE_ZYX,
)

napari.run()
